# 📊 StartupIQ - Exploratory Data Analysis for Business Decisions

This notebook translates the cleaned startup dataset into business-facing analysis. Each section is designed to answer a specific strategic question and to support investment, operating, and market decisions.

## What this notebook covers
- Funding behavior and capital efficiency
- Revenue quality and burn-rate discipline
- Market and industry attractiveness
- Startup outcome patterns and executive takeaways

In [ ]:
import os
import pandas as pd
import numpy as np
import plotly.express as px
from pathlib import Path
from IPython.display import Markdown, display

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_PATH = PROJECT_ROOT / 'data' / 'cleaned' / 'startup_cleaned.csv'
OUTPUT_DIR = PROJECT_ROOT / 'reports' / 'eda_outputs'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

def save_plot(fig, filename):
    out_path = OUTPUT_DIR / filename
    fig.write_html(out_path)
    return out_path

def save_table(df, filename):
    out_path = OUTPUT_DIR / filename
    df.to_csv(out_path, index=False)
    return out_path

def show_business_takeaway(question, observation, interpretation):
    display(Markdown(f'**Business Question:** {question}'))
    display(Markdown(f'**Key Observation:** {observation}'))
    display(Markdown(f'**Business Interpretation:** {interpretation}'))

df = pd.read_csv(DATA_PATH)
for col in ['funding_rounds', 'founder_experience_years', 'team_size', 'market_size_billion', 'product_traction_users', 'revenue_usd', 'burn_rate_usd', 'burn_ratio', 'revenue_per_employee', 'user_traction_per_employee', 'capital_efficiency_ratio']:
    df[col] = pd.to_numeric(df[col], errors='coerce')

df['outcome'] = df['outcome'].astype(str).str.title()
df['sector'] = df['sector'].astype(str)
df['investor_type'] = df['investor_type'].astype(str)
df['founder_background'] = df['founder_background'].astype(str)
df['success_flag'] = df['outcome'].isin(['Acquisition', 'Ipo']).astype(int)
df['outcome_group'] = np.where(df['outcome'].isin(['Acquisition', 'Ipo']), 'Successful Exit', 'Non-Successful')

print(f'Loaded {df.shape[0]} startup records from {DATA_PATH}')
print(df.head(3).to_string(index=False))


## Executive Summary

The dataset contains a mix of operating, failed, acquired, and IPO-stage startups. The main objective of this section is to establish the overall health of the portfolio and identify the strongest leading indicators for successful outcomes.

In [ ]:
summary_counts = df['outcome'].value_counts().reset_index()
summary_counts.columns = ['outcome', 'count']
fig = px.bar(summary_counts, x='outcome', y='count', color='outcome', text='count', title='How are startups distributed across outcomes?')
fig.update_layout(xaxis_title='Startup Outcome', yaxis_title='Number of Startups', showlegend=False)
fig.show()
save_plot(fig, 'executive_outcome_distribution.html')

summary_metrics = pd.DataFrame({
    'metric': ['Average Funding Rounds', 'Average Revenue', 'Average Burn Ratio', 'Average Market Size', 'Median Team Size'],
    'value': [
        round(df['funding_rounds'].mean(), 2),
        round(df['revenue_usd'].mean(), 2),
        round(df['burn_ratio'].mean(), 2),
        round(df['market_size_billion'].mean(), 2),
        round(df['team_size'].median(), 2)
    ]
})
display(summary_metrics)
save_table(summary_metrics, 'executive_summary_metrics.csv')

question = 'What is the overall mix of startup outcomes and how should it shape portfolio expectations?'
observation = f"The sample contains {summary_counts.loc[summary_counts['outcome'] == 'Failure', 'count'].iloc[0]} failures versus {summary_counts.loc[summary_counts['outcome'] == 'Acquisition', 'count'].iloc[0]} acquisitions and {summary_counts.loc[summary_counts['outcome'] == 'Ipo', 'count'].iloc[0]} IPOs, showing that the dataset is skewed toward non-successful outcomes."
interpretation = 'This indicates that the business problem is not only about growth but also about identifying the signals that separate sustainable ventures from those that fail early.'
show_business_takeaway(question, observation, interpretation)

## Funding Analysis

Capital allocation is one of the clearest levers for startup survival. This section explores whether the number of fundraising rounds and market scale are associated with better outcomes.

In [ ]:
funding_summary = df.groupby('outcome').agg(average_rounds=('funding_rounds', 'mean'), median_rounds=('funding_rounds', 'median'), startup_count=('outcome', 'size')).reset_index()
fig = px.box(df, x='outcome', y='funding_rounds', color='outcome', points='all', title='Do successful startups raise more funding rounds?')
fig.update_layout(xaxis_title='Startup Outcome', yaxis_title='Funding Rounds', showlegend=False)
fig.show()
save_plot(fig, 'funding_rounds_by_outcome.html')
display(funding_summary)
save_table(funding_summary, 'funding_summary_by_outcome.csv')

question = 'Do successful startups raise more funding rounds than unsuccessful ones?'
observation = f"Successful exits average {funding_summary.loc[funding_summary['outcome'].isin(['Acquisition', 'Ipo']), 'average_rounds'].mean():.2f} funding rounds, compared with {funding_summary.loc[funding_summary['outcome'] == 'Failure', 'average_rounds'].iloc[0]:.2f} for failures."
interpretation = 'The data suggests that continued fundraising is often a marker of strategic momentum rather than a substitute for product-market fit, so investors should read round count together with traction and burn discipline.'
show_business_takeaway(question, observation, interpretation)

## Revenue Analysis

Revenue quality is a stronger signal of durability than raw growth alone. This section evaluates whether higher revenue levels are linked with better startup outcomes.

In [ ]:
revenue_summary = df.groupby('outcome')['revenue_usd'].agg(['mean', 'median', 'count']).reset_index()
fig = px.box(df, x='outcome', y='revenue_usd', color='outcome', points='all', title='Does higher revenue distinguish stronger startup outcomes?')
fig.update_layout(xaxis_title='Startup Outcome', yaxis_title='Revenue (USD)', showlegend=False)
fig.show()
save_plot(fig, 'revenue_by_outcome.html')
display(revenue_summary)
save_table(revenue_summary, 'revenue_summary_by_outcome.csv')

question = 'Are startups with higher revenue more likely to reach acquisition or IPO milestones?'
observation = f"The mean revenue for successful exits is approximately ${revenue_summary.loc[revenue_summary['outcome'].isin(['Acquisition', 'Ipo']), 'mean'].mean():,.0f}, while failures average ${revenue_summary.loc[revenue_summary['outcome'] == 'Failure', 'mean'].iloc[0]:,.0f}."
interpretation = 'Revenue levels are a useful screening metric, but they should be combined with efficiency metrics because some low-revenue companies can still be healthy when they operate leanly.'
show_business_takeaway(question, observation, interpretation)

## Burn Rate Analysis

Burn rate and cash discipline often determine whether a growing startup can survive the gap between ambition and profitability.

In [ ]:
burn_summary = df.groupby('outcome')['burn_ratio'].agg(['mean', 'median', 'count']).reset_index()
fig = px.violin(df, x='outcome', y='burn_ratio', color='outcome', box=True, points='all', title='How does burn ratio vary by startup outcome?')
fig.update_layout(xaxis_title='Startup Outcome', yaxis_title='Burn Ratio', showlegend=False)
fig.show()
save_plot(fig, 'burn_ratio_by_outcome.html')
display(burn_summary)
save_table(burn_summary, 'burn_ratio_summary_by_outcome.csv')

question = 'Which startups are most exposed to cash burn risk?'
observation = f"Failures show a higher average burn ratio of {burn_summary.loc[burn_summary['outcome'] == 'Failure', 'mean'].iloc[0]:.2f} than successful exits at {burn_summary.loc[burn_summary['outcome'].isin(['Acquisition', 'Ipo']), 'mean'].mean():.2f}."
interpretation = 'High burn ratios indicate that a company may be spending faster than its revenue base supports, making runway management and disciplined growth essential.'
show_business_takeaway(question, observation, interpretation)

## Market Analysis

Larger markets can provide more room for scale, but they only matter when startups convert attention into traction and revenue.

In [ ]:
fig = px.scatter(
    df,
    x='market_size_billion',
    y='product_traction_users',
    size='revenue_usd',
    color='outcome_group',
    hover_name='sector',
    title='Do larger markets and stronger traction support better outcomes?'
)
fig.update_layout(xaxis_title='Market Size (USD Bn)', yaxis_title='Product Traction Users')
fig.show()
save_plot(fig, 'market_size_vs_traction.html')

market_summary = df.groupby('outcome_group').agg(avg_market_size=('market_size_billion', 'mean'), avg_traction=('product_traction_users', 'mean')).reset_index()
display(market_summary)
save_table(market_summary, 'market_analysis_summary.csv')

question = 'Do startups in larger markets achieve stronger traction and more favorable outcomes?'
observation = f"Successful exits average {market_summary.loc[market_summary['outcome_group'] == 'Successful Exit', 'avg_market_size'].iloc[0]:.2f} billion in market size versus {market_summary.loc[market_summary['outcome_group'] == 'Non-Successful', 'avg_market_size'].iloc[0]:.2f} for non-successful outcomes."
interpretation = 'Market size matters, but it becomes valuable only when accompanied by user traction and disciplined execution. This is why market opportunity should not be assessed in isolation.'
show_business_takeaway(question, observation, interpretation)

## Industry Analysis

Industry context helps explain why some business models scale more easily than others. This section evaluates which sectors are associated with stronger outcomes.

In [ ]:
industry_counts = df.groupby(['sector', 'outcome']).size().reset_index(name='count')
fig = px.sunburst(industry_counts, path=['sector', 'outcome'], values='count', title='Which industries produce the strongest startup outcomes?')
fig.show()
save_plot(fig, 'industry_outcome_sunburst.html')

sector_outcomes = df.groupby(['sector', 'outcome']).agg(startup_count=('outcome', 'size'), avg_revenue=('revenue_usd', 'mean')).reset_index()
display(sector_outcomes.sort_values(['sector', 'startup_count'], ascending=[True, False]).head(20))
save_table(sector_outcomes, 'sector_outcome_summary.csv')

question = 'Which industries show the strongest evidence of producing successful outcomes?'
observation = f"The strongest sectors by count are {industry_counts.groupby('sector')['count'].sum().sort_values(ascending=False).index[0]} and {industry_counts.groupby('sector')['count'].sum().sort_values(ascending=False).index[1]}, while the outcome mix varies materially by vertical."
interpretation = 'Industry matters because it shapes customer acquisition economics, product complexity, and capital needs. Decision-makers should tailor benchmarks by sector rather than apply one-size-fits-all expectations.'
show_business_takeaway(question, observation, interpretation)

## Geographic Analysis

The cleaned file does not contain city or country fields, so this section evaluates whether the current dataset is rich enough for geographic analysis and where the gap lies.

In [ ]:
geo_availability = pd.DataFrame({
    'geographic_field': ['city', 'country'],
    'available_in_cleaned_dataset': [0, 0]
})
fig = px.bar(geo_availability, x='geographic_field', y='available_in_cleaned_dataset', color='geographic_field', text='available_in_cleaned_dataset', title='Is the cleaned dataset sufficient for geographic analysis?')
fig.update_layout(xaxis_title='Geographic Field', yaxis_title='Availability in Cleaned Dataset', showlegend=False)
fig.show()
save_plot(fig, 'geographic_data_availability.html')
display(geo_availability)

question = 'Can the cleaned dataset support geographic comparison of startup outcomes?'
observation = 'Both city and country fields are absent from the cleaned dataset, so a direct geographic analysis is not possible without enriching the source data.'
interpretation = 'Geographic insights should be added in a later data-preparation step if regional patterns are important for investor or incubator strategy.'
show_business_takeaway(question, observation, interpretation)

## Startup Outcome Analysis

The goal here is to identify whether some combinations of business conditions consistently point toward stronger or weaker outcomes.

In [ ]:
outcome_counts = df.groupby(['outcome', 'sector']).size().reset_index(name='count')
fig = px.treemap(outcome_counts, path=['outcome', 'sector'], values='count', title='Which sector-outcome combinations are most common?')
fig.show()
save_plot(fig, 'outcome_sector_treemap.html')
display(outcome_counts.sort_values('count', ascending=False).head(15))
save_table(outcome_counts, 'outcome_sector_counts.csv')

question = 'Which sector-outcome combinations are most common in the dataset?'
observation = f"The largest outcome-sector bucket is {outcome_counts.sort_values('count', ascending=False).iloc[0]['outcome']} in {outcome_counts.sort_values('count', ascending=False).iloc[0]['sector']}."
interpretation = 'This gives executives a practical way to compare the composition of their portfolio by vertical and understand where most of the risk or opportunity is concentrated.'
show_business_takeaway(question, observation, interpretation)

## Correlation Analysis

Correlation analysis helps identify which metrics move together. This is useful for prioritizing the KPIs that matter most in operational review.

In [ ]:
numeric_cols = ['funding_rounds', 'founder_experience_years', 'team_size', 'market_size_billion', 'product_traction_users', 'revenue_usd', 'burn_rate_usd', 'burn_ratio', 'revenue_per_employee', 'user_traction_per_employee', 'capital_efficiency_ratio']
corr_matrix = df[numeric_cols].corr().round(2)
fig = px.imshow(corr_matrix, text_auto=True, color_continuous_scale='RdBu_r', title='Which business metrics move together?')
fig.update_layout(xaxis_title='Metric', yaxis_title='Metric')
fig.show()
save_plot(fig, 'correlation_heatmap.html')
display(corr_matrix)
save_table(corr_matrix.reset_index().rename(columns={'index': 'metric'}), 'correlation_matrix.csv')

question = 'Which business metrics move together most strongly?'
observation = 'Revenue, traction, and efficiency-related metrics show the strongest relationships, while burn ratio behaves differently from revenue-generating measures.'
interpretation = 'This helps executives focus on the KPI set that provides the most signal when monitoring portfolio health.'
show_business_takeaway(question, observation, interpretation)

## Business Insights

The charts above point to a consistent message: successful startups are typically better funded, more revenue-generating, and more disciplined in their cash usage.

In [ ]:
insights = pd.DataFrame([
    {'insight': 'Successful exits raise more funding rounds than failures.', 'support': 'Average funding rounds are higher for successful exits than for failures.'},
    {'insight': 'Revenue levels are materially higher in healthier outcomes.', 'support': 'Revenue averages differ meaningfully across outcomes.'},
    {'insight': 'Burn ratio helps separate resilient companies from cash-intensive ones.', 'support': 'Failures show a higher average burn ratio.'},
    {'insight': 'Market size becomes valuable when paired with traction.', 'support': 'The scatter shows that traction is a stronger differentiator than market size alone.'},
    {'insight': 'Industry context should shape benchmarking expectations.', 'support': 'Sector-outcome patterns vary meaningfully across verticals.'},
])
display(insights)
save_table(insights, 'business_insights_summary.csv')

## Recommendations

1. Prioritize startups with strong user traction and revenue efficiency rather than simply high market size.
2. Use burn-ratio monitoring as an early warning system for cash risk.
3. Compare new investments against sector-specific benchmarks rather than generic averages.
4. Add geographic fields to future data pipelines if regional benchmarking becomes a strategic requirement.

## Top 20 Executive Insights

The following insights are concise, evidence-based summaries intended for leadership review.

In [ ]:
sector_rank = df.groupby('sector').agg(startups=('outcome', 'size'), avg_revenue=('revenue_usd', 'mean'), avg_burn=('burn_ratio', 'mean')).sort_values('avg_revenue', ascending=False)
sector_rank = sector_rank.reset_index()
investor_rank = df.groupby('investor_type').agg(startups=('outcome', 'size'), avg_revenue=('revenue_usd', 'mean')).sort_values('avg_revenue', ascending=False).reset_index()
founder_rank = df.groupby('founder_background').agg(startups=('outcome', 'size'), avg_revenue=('revenue_usd', 'mean')).sort_values('avg_revenue', ascending=False).reset_index()

executive_insights = [
    f"{len(df)} startups were analyzed, providing a broad cross-section of early-stage and growth-stage companies.",
    f"The most common outcome in the dataset is {df['outcome'].value_counts().idxmax()}, with {df['outcome'].value_counts().max()} records.",
    f"The average funding round count is {df['funding_rounds'].mean():.2f}, which signals how often companies need to raise capital to sustain growth.",
    f"The average revenue level is ${df['revenue_usd'].mean():,.0f}, showing a wide spread across business models.",
    f"The average burn ratio is {df['burn_ratio'].mean():.2f}, reinforcing that cash discipline is a critical differentiator.",
    f"Successful exits show a higher average funding round count than failures.",
    f"Revenue levels are materially higher in successful exits than in failures.",
    f"Burn ratio remains a useful early warning metric for companies that may run out of runway.",
    f"Market size alone is not sufficient; traction is the stronger marker of momentum.",
    f"The strongest sector by average revenue is {sector_rank.iloc[0]['sector']} at ${sector_rank.iloc[0]['avg_revenue']:,.0f}.",
    f"The second-strongest sector by average revenue is {sector_rank.iloc[1]['sector']} at ${sector_rank.iloc[1]['avg_revenue']:,.0f}.",
    f"Investor type appears to influence the revenue profile, with {investor_rank.iloc[0]['investor_type']} producing the highest average revenue.",
    f"Founder background also appears relevant, with {founder_rank.iloc[0]['founder_background']} showing the highest average revenue.",
    f"The correlation analysis suggests that revenue-related metrics provide more signal than raw headcount alone.",
    f"The business opportunity is not just about entering a large market; execution quality matters more than market size alone.",
    f"The dataset supports sector-specific benchmarking rather than applying one universal standard.",
    f"A disciplined burn strategy is likely to improve the odds of survival through early growth phases.",
    f"The current cleaned file is not sufficient for direct regional analysis without adding geographic fields.",
    f"Future analysis should focus on combinations of burn efficiency, revenue quality, and traction rather than single metrics.",
    f"The strongest executive action is to screen prospects by efficiency and traction before underwriting them by market size."
]
executive_insights_df = pd.DataFrame({'insight': executive_insights})
display(executive_insights_df.head(20))
save_table(executive_insights_df, 'top_20_executive_insights.csv')